# C1 Session 2 — Generalization: Train/Test Discipline and Overfitting

*Machine Learning Fundamentals, session 2 of 3 (~85 min). Prerequisites:
F1-scientific-python and Session 1.*

Supervised learning promises rules that work on **new** data. This session
builds the machinery that keeps that promise honest — the train/test split —
and then stages the classic disaster (a rule that aces training and fails
reality) closely enough to dissect it. Answers to all checkpoints are at the
end of this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 20260804
rng = np.random.default_rng(SEED)

## 1. Why Honest Measurement Needs Held-Out Data

**Motivation.** Suppose a student memorizes the answer key of a past exam and
then "takes" that same exam, scoring 100%. Have they learned the subject?
Obviously not — the only honest test is one with **questions they have never
seen**.

Learned rules deserve the same suspicion. A rule can look brilliant on the
examples it was built from and still be useless on new data — because scoring
a rule on its own training examples measures *absorption*, not *ability*. So
we split our labeled data **before** doing anything else:

- the **training set** — the examples the rule is allowed to learn from;
- the **test set** — locked in a drawer, never touched while building the
  rule, used exactly once at the end to measure honest performance.

From here on, one measurement matters most: **accuracy on the test set** —
the fraction of test examples the rule gets right.

### Checkpoint 1

1. Why is the memorizing student's 100% not a measure of learning, and what
   is the machine-learning equivalent of "the same exam again"?
2. A friend evaluates on the test set, tweaks the rule, evaluates again,
   tweaks again — twenty times — and reports the best test score. Why is
   that score no longer honest?

## 2. The Seeded Shuffle-and-Split, by Hand

**Worked example.** Two rules of craftsmanship, both from
F1-scientific-python: *seed the shuffle* so the split is reproducible, and
*apply the same permutation to inputs and labels* so each input stays glued
to its own label.

In [ ]:
X = np.array([4.2, 6.8, 3.9, 7.1, 5.0, 6.4, 4.7, 7.6, 3.5, 6.0, 5.4, 7.3])
y = np.array([0,   1,   0,   1,   0,   1,   0,   1,   0,   1,   0,   1])

split_rng = np.random.default_rng(SEED)
order = split_rng.permutation(len(X))   # a shuffled list of the indices 0..11
print("shuffled order:", order)

X_shuf, y_shuf = X[order], y[order]     # SAME order for both arrays

n_train = 9                              # 9 train / 3 test
X_train, y_train = X_shuf[:n_train], y_shuf[:n_train]
X_test,  y_test  = X_shuf[n_train:], y_shuf[n_train:]

print("train inputs:", X_train)
print("train labels:", y_train)
print("test inputs: ", X_test)
print("test labels: ", y_test)

Why shuffle at all? Data often arrives in a meaningful order (all class-0
examples first, say). Slicing without shuffling would put every class-0
example in training and none in test. The seeded permutation deals the
examples out fairly — and reproducibly: anyone running this notebook gets the
identical split.

### Checkpoint 2

1. Why must `X` and `y` be reordered by the *same* permutation? Describe what
   breaks if you shuffle them independently.
2. You shuffle 12 labeled examples with a seeded permutation and split 75/25.
   What are the shapes of the four resulting arrays, and why did the shuffle
   have to happen before the slicing?

## 3. A Dataset and the Memorizer

**Motivation.** Now we stage the classic disaster. We will build two
competing classifiers for the same task and watch one of them ace the
training set while failing the real test.

**The data.** One measurement per beetle (body length in mm) and a species
label: species `0` runs small, species `1` runs large, with overlap in the
middle. Seeded generation, then a shuffle-and-split exactly as in Section 2:

In [ ]:
data_rng = np.random.default_rng(SEED)
length_0 = data_rng.normal(3.0, 1.0, size=40)   # species 0: centered near 3 mm
length_1 = data_rng.normal(6.0, 1.0, size=40)   # species 1: centered near 6 mm

X = np.concatenate([length_0, length_1])
y = np.concatenate([np.zeros(40, dtype=int), np.ones(40, dtype=int)])

order = data_rng.permutation(len(X))
X, y = X[order], y[order]
X_train, y_train = X[:60], y[:60]
X_test,  y_test  = X[60:], y[60:]
print("train size:", len(X_train), " test size:", len(X_test))

*Plotting note:* `fig, ax = plt.subplots()` is matplotlib's two-object form of the `plt.*` calls covered in F1-scientific-python — `ax.scatter`, `ax.set_xlabel`, and `ax.axvline` do exactly what `plt.scatter`, `plt.xlabel`, and `plt.axvline` do, on an explicit figure-and-axes pair.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.6))
ax.scatter(X_train[y_train == 0], np.zeros(np.sum(y_train == 0)),
           marker="o", s=42, color="#2a78d6", label="species 0")
ax.scatter(X_train[y_train == 1], np.ones(np.sum(y_train == 1)),
           marker="^", s=42, color="#eb6834", label="species 1")
ax.set_yticks([0, 1])
ax.set_yticklabels(["species 0", "species 1"])
ax.set_xlabel("body length (mm)")
ax.set_title("Training data: one measurement per beetle")
ax.legend(loc="center right")
plt.show()

**Classifier 1: the memorizer.** A lookup table. It stores every training
measurement together with its label. Asked about a length it has seen, it
answers from the table; asked about anything else, it shrugs and guesses the
most common training label:

In [ ]:
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def memorizer_predict(X_tr, y_tr, X_new):
    # np.bincount counts occurrences of each small non-negative int
    # (here: how many 0s and how many 1s); argmax picks the commoner label.
    majority = int(np.argmax(np.bincount(y_tr)))          # fallback guess
    table = {float(v): int(lab) for v, lab in zip(X_tr, y_tr)}
    return np.array([table.get(float(v), majority) for v in X_new])

mem_train_acc = accuracy(y_train, memorizer_predict(X_train, y_train, X_train))
mem_test_acc  = accuracy(y_test,  memorizer_predict(X_train, y_train, X_test))
print("memorizer  train accuracy:", mem_train_acc)
print("memorizer  test  accuracy:", mem_test_acc)

Perfect on training — of course: it *is* the training set, written out as a
table. But the test lengths are new measurements it has never seen, so every
single test answer falls back to the majority guess. The memorizer learned
the examples and not the pattern.

### Checkpoint 3

1. Why does the memorizer's *perfect* training score carry no information
   about how good it is? (Hint: could any training set make it score below
   100%?)
2. What exactly does the memorizer answer for an input it never stored — and
   which simple rule does that make it identical to on fully unseen data?

## 4. The Threshold Rule and the Scoreboard

**Classifier 2: a single threshold.** The humblest possible rule: *pick one
cutoff `t` and predict species 1 whenever the length is at least `t`.* (The
species-1 beetles run larger, so "at least" is the sensible direction.) We
try every midpoint between neighbouring sorted training lengths and keep the
cutoff that gets the most training examples right:

In [ ]:
def choose_threshold(X_tr, y_tr):
    s = np.sort(X_tr)
    candidates = (s[:-1] + s[1:]) / 2        # midpoints between neighbours
    best_t, best_acc = candidates[0], -1.0
    for t in candidates:
        acc = np.mean((X_tr >= t).astype(int) == y_tr)
        if acc > best_acc:
            best_t, best_acc = float(t), float(acc)
    return best_t

def threshold_predict(t, X_new):
    return (X_new >= t).astype(int)

t = choose_threshold(X_train, y_train)
thr_train_acc = accuracy(y_train, threshold_predict(t, X_train))
thr_test_acc  = accuracy(y_test,  threshold_predict(t, X_test))
print(f"chosen cutoff: {t:.3f} mm")
print("threshold  train accuracy:", thr_train_acc)
print("threshold  test  accuracy:", thr_test_acc)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.6))
ax.scatter(X_train[y_train == 0], np.zeros(np.sum(y_train == 0)),
           marker="o", s=42, color="#2a78d6", label="species 0")
ax.scatter(X_train[y_train == 1], np.ones(np.sum(y_train == 1)),
           marker="^", s=42, color="#eb6834", label="species 1")
ax.axvline(t, color="#555555", linewidth=2, linestyle="--",
           label=f"cutoff = {t:.2f} mm")
ax.set_yticks([0, 1])
ax.set_yticklabels(["species 0", "species 1"])
ax.set_xlabel("body length (mm)")
ax.set_title("The single-threshold rule on the training data")
ax.legend(loc="center right")
plt.show()

**The scoreboard.** The memorizer *wins on training* and *falls apart on
test*. The threshold rule concedes a few training mistakes (the overlapping
beetles in the middle are genuinely ambiguous) but keeps nearly all of its
accuracy on data it never saw.

This failure pattern has a name. A rule **overfits** when it performs much
better on its training examples than on new examples, because it latched onto
the specific examples it was shown instead of the pattern behind them. The
memorizer is overfitting in its purest form: a huge gap between train and
test accuracy. The gap is the tell — always compare the two numbers.

### Checkpoint 4

1. A classifier scores 98% on training data and 71% on test data. What is
   this phenomenon called, and which number better predicts real-world
   performance?
2. The threshold rule concedes training mistakes the memorizer does not. Why
   is that concession a strength here rather than a weakness?

## 5. Flexible Rules, Rigid Rules

**Motivation.** Why did the memorizer fail while the threshold survived? It
is not about lookup tables specifically — it is about **how flexible** a rule
is allowed to be. Picture a spectrum:

- **Too rigid.** *"Always predict species 1, whatever the input."* This rule
  cannot bend to the data at all. It makes the same systematic mistake on
  every dataset — every species-0 beetle is missed — and no amount of extra
  training data fixes it.
- **Too flexible.** The memorizer. It bends to *every* detail of the training
  examples, including meaningless ones. The rule it walks away with *is* the
  training set — change the sample and the entire stored table changes with
  it.
- **In between.** The threshold rule. Flexible enough to place its cutoff
  where the data says, rigid enough to ignore individual quirks.

**Worked example: five re-deals, train and test side by side.** Let's
re-deal the same 80 beetles into different train/test splits five times and,
for each rule, print its training accuracy next to its test accuracy — plus
the threshold's chosen cutoff. The train/test *pair* is what separates the
three rules:

In [ ]:
print(" deal | rigid tr/te | thresh tr/te | memor tr/te | cutoff")
cutoffs = []
for k in range(5):
    deal = np.random.default_rng(SEED + k).permutation(len(X))
    Xs, ys = X[deal], y[deal]
    Xtr, ytr, Xte, yte = Xs[:60], ys[:60], Xs[60:], ys[60:]

    rigid_tr = accuracy(ytr, np.ones(len(ytr), dtype=int))   # always species 1
    rigid_te = accuracy(yte, np.ones(len(yte), dtype=int))
    tk = choose_threshold(Xtr, ytr)
    thr_tr = accuracy(ytr, threshold_predict(tk, Xtr))
    thr_te = accuracy(yte, threshold_predict(tk, Xte))
    mem_tr = accuracy(ytr, memorizer_predict(Xtr, ytr, Xtr))
    mem_te = accuracy(yte, memorizer_predict(Xtr, ytr, Xte))
    cutoffs.append(tk)
    print(f"  {k}   | {rigid_tr:.2f} / {rigid_te:.2f} | {thr_tr:.2f} / {thr_te:.2f}  "
          f"| {mem_tr:.2f} / {mem_te:.2f} | {tk:.2f} mm")

print("cutoff moves only within:", f"{min(cutoffs):.2f}-{max(cutoffs):.2f} mm")

Read the table pair by pair.

- **Rigid rule:** train and test are both mediocre *and close together*
  (either side of 0.5). It commits to the same systematic mistake on every
  deal; nothing about it depends on the sample, and nothing about it
  improves.
- **Memorizer:** training accuracy is a perfect 1.00 on *every* deal — its
  stored table simply is whichever 60 beetles it was dealt, so the "rule" it
  ends up with changes wholesale from deal to deal. Yet its *test* accuracy
  sits at rigid-rule level every time: on measurements it never stored it
  falls back to the majority guess (Checkpoint 3), so none of the perfectly
  memorized detail transfers. The evidence of its excess flexibility is the
  train column read against the test column — a 1.00-to-roughly-0.4 gap on
  every single deal.
- **Threshold:** strong on both sides of the slash with only a small gap, and
  its learned cutoff barely moves between deals (4.32–4.60 mm) — the rule
  found something *stable* about beetles, not something about these
  particular beetles.

So when a rule underperforms, ask which side of the spectrum it sits on:

- **Too rigid** → it misses real pattern. Symptom: train and test accuracy
  are *both* poor, and close together.
- **Too flexible** → what it learns is a portrait of its particular training
  sample (the memorizer's table changes completely with every deal), and the
  absorbed detail does not transfer. Symptom: the overfitting gap — great
  train, poor test, on every deal.

> **Scope note.** This flexible-versus-rigid tradeoff has a formal name in
> statistics: the *bias–variance tradeoff*. Making "bias" and "variance"
> precise as defined quantities takes statistical machinery that arrives in
> later units. In this unit we deliberately stay with the everyday meaning:
> rigid rules err systematically, and flexible rules give answers that vary
> a lot depending on which training examples they happened to see.

### Checkpoint 5

1. Place these three rules on the rigid–flexible spectrum: (a) always
   predict the majority class; (b) the single threshold; (c) the memorizer.
2. One rule's *learned content* is nothing but the particular training
   sample it was dealt. Which rule, which column pair of the table proves
   that this buys nothing transferable, and what do its predictions reduce
   to on fully unseen measurements?

## 6. Common Pitfalls (Discipline Edition)

**Pitfall 1: shuffling `X` and `y` independently.** Two different
permutations tear inputs away from their own labels. Broken:

In [ ]:
# BROKEN: two different generators -> two different orders
r1 = np.random.default_rng(1)
r2 = np.random.default_rng(2)
X_bad = X[r1.permutation(len(X))]
y_bad = y[r2.permutation(len(y))]

# How many inputs still carry their own label?
original_label = {float(v): int(lab) for v, lab in zip(X, y)}
still_paired = np.mean([original_label[float(v)] == lab
                        for v, lab in zip(X_bad, y_bad)])
print("fraction of pairs still intact:", still_paired)

In [ ]:
# FIX: one permutation, applied to both arrays
order = np.random.default_rng(1).permutation(len(X))
X_good, y_good = X[order], y[order]
still_paired = np.mean([original_label[float(v)] == lab
                        for v, lab in zip(X_good, y_good)])
print("fraction of pairs still intact:", still_paired)

With independent shuffles roughly half the "dataset" is silently mislabeled —
and nothing crashes. Any rule learned from it is meaningless.

**Pitfall 2: letting the rule see the test set.** Any information flowing
from the test set into rule-building quietly turns test data into training
data. The broken example "fixes" the memorizer by letting it memorize *all*
the data — and its test score becomes perfect:

In [ ]:
# BROKEN: the lookup table is built from train AND test examples.
X_all = np.concatenate([X_train, X_test])
y_all = np.concatenate([y_train, y_test])
leaky_acc = accuracy(y_test, memorizer_predict(X_all, y_all, X_test))
print("memorizer test accuracy with leakage:", leaky_acc)   # a perfect illusion

# FIX: the rule may only ever see the training set.
honest_acc = accuracy(y_test, memorizer_predict(X_train, y_train, X_test))
print("memorizer test accuracy, honest:     ", honest_acc)

The leaky 1.0 is pure illusion: the memorizer did not get better — the exam
questions were slipped into its notes. Leakage is rarely this blatant; it
usually arrives as repeated peeking (Checkpoint 1) or as preprocessing
computed on all the data before splitting. The rule is absolute: **nothing
about the test set may influence the rule before the final measurement.**

**Pitfall 3: choosing between rules by training accuracy.** Training scores
always flatter the flexible candidate. Broken: "the memorizer beats the
threshold 1.00 to 0.93 on training — deploy the memorizer." Fix: compare
candidates on data neither has seen — hold back a slice of the *training*
portion for the comparison, and keep the test set untouched for the final
measurement:

In [ ]:
# FIX in code: split the training portion 45/15 for candidate comparison
X_sub, y_sub   = X_train[:45], y_train[:45]
X_hold, y_hold = X_train[45:], y_train[45:]

mem_hold = accuracy(y_hold, memorizer_predict(X_sub, y_sub, X_hold))
t_sub    = choose_threshold(X_sub, y_sub)
thr_hold = accuracy(y_hold, threshold_predict(t_sub, X_hold))
print("held-back comparison -- memorizer:", round(mem_hold, 3),
      " threshold:", round(thr_hold, 3))
# The threshold wins where it counts; the test set was never touched.

### Checkpoint 6

1. Spot the leak: a teammate computes the list of candidate cutoffs from all
   80 beetles (train and test together), then picks the best on training
   data only and evaluates on test. Is the final number honest?
2. A teammate never concatenates train and test arrays, but re-runs the test
   evaluation after each of 30 tweaks and keeps the best-scoring variant.
   Explain why this is still test-set leakage.

## 7. Exam Connections and Going Deeper

Round 1's "ML concepts" cluster (see `reference/analysis.md`,
topic-distribution table) tests this session's material with five-option
concept MC items: reading a train/test gap, recognizing overfitting, and
judging accuracies against the always-majority baseline — all at paraphrase
level, no code required.

**Worked exam-style example.** *Reasoning is required. No coding is needed.*

A handwriting classifier scores **96%** accuracy on its training set and
**62%** on the held-out test set. Predicting the most common letter for
every input would score **65%** on that same test set. Which statement best
describes the situation?

- **A.** The rule is too rigid; it should be made flexible enough to absorb
  more detail from the training examples.
- **B.** The test set must be defective, because training accuracy is the
  truer measure of a rule's quality.
- **C.** Nothing is wrong: 96% training accuracy shows the pattern has been
  learned.
- **D.** The rule overfits, but since 62% is close to 65% it effectively
  matches the baseline and is fine to ship.
- **E.** The rule is too flexible: the large train-test gap is overfitting,
  and scoring *below* the 65% always-majority baseline means it currently
  does worse than a rule that ignores its input entirely.

*Worked solution.* Step 1 — read the gap: 96% vs 62% is a large train-test
gap, the signature of an over-flexible rule, eliminating A (more flexibility
worsens it) and C (training accuracy proves absorption, not ability). Step 2
— trust direction: the test set is the honest measurement, eliminating B.
Step 3 — the baseline: 62% < 65%, so the rule loses to the zero-effort
constant rule; "close to the baseline" is not a pass mark, eliminating D.
Answer: **E**. Note the register: every wrong option encodes a specific
misconception, and the baseline comparison is the discriminating fact.

**Going deeper (optional).** **C2-linear-models** builds rules with many
learned numbers instead of one cutoff — and with principled ways to keep
that flexibility from tipping into overfitting. **F5-probability** supplies
the tools that make "how much does a score move from sample to sample"
precise, turning Section 5's spectrum into defined quantities. Neither is
needed for this unit's practice.

### Checkpoint 7

1. In the worked example, why is the comparison against the always-majority
   baseline the decisive fact, on the exam and in practice?
2. Exam coding parts fix exact function names and contracts. What does that
   exactness let the grader do that a loose description would not?

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. The score measures recall of stored answers, not ability to answer new
   questions. The ML equivalent is evaluating a rule on its own training
   examples.
2. By tweaking twenty times against the test set, the friend has been *using
   the test set to build the rule* — it has quietly become training data.
   The reported best score is tuned to those particular examples and will
   not hold on genuinely new data.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. Each label belongs to one specific input. Shuffling `X` and `y` with
   different permutations pairs inputs with other examples' labels, so the
   "dataset" becomes scrambled nonsense — and any rule learned from it is
   meaningless.
2. Shapes `(9,)`, `(9,)`, `(3,)`, `(3,)`. The shuffle must come first
   because the data may arrive ordered (e.g. by class); slicing an ordered
   array would give train and test sets with different class mixtures.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. No training set can make the memorizer score below 100%: it repeats
   stored labels back. A score that *cannot* come out low regardless of
   whether the rule generalizes carries no information about
   generalization.
2. It answers the most common training label. On fully unseen data it is
   therefore identical to the always-majority constant rule.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. Overfitting. The 71% test score is the better predictor of real-world
   performance — new data is exactly what the test set rehearses.
2. The conceded examples sit in the genuinely ambiguous overlap zone.
   Refusing to contort around them is what lets the rule capture the stable
   pattern — the mistakes it accepts on training are the price of accuracy
   that transfers.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. Rigid end: (a) always-majority. Middle: (b) the threshold. Flexible end:
   (c) the memorizer.
2. The memorizer — its stored table *is* the training sample, so its
   learned content changes wholesale with every deal. The proof is the
   train/test pair: 1.00 training accuracy on every deal against test
   accuracy stuck at the rigid rule's level, because on unseen measurements
   it degenerates to the majority guess (Checkpoint 3). The gap between
   those two columns is the overfitting signature.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. No. The candidate list was computed from test measurements, so the test
   set influenced which rules were even considered — information leaked
   before the final measurement, and the reported number is gently inflated.
2. Each evaluation leaks information: keeping the variant that scored best
   *on the test set* uses the test set to choose the rule, exactly as
   training data would be used. After 30 selections the reported score is
   tuned to those particular test examples and overstates performance on
   genuinely new data.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. The baseline is the score of a rule that does nothing clever. Any
   candidate below it has negative value — its learned "pattern" is worse
   than ignoring the input — so the comparison instantly converts raw
   accuracy into a meaningful judgment.
2. Exact contracts make the solution machine-checkable: the grader can call
   the named function with hidden inputs and compare outputs, with no
   judgment calls about what the student "meant".

</details>